# EduReport: Análisis de Datos Escolares y Verificación de Lógica de Negocio

**Proyecto:** EduReport - Sistema de Gestión Docente y Automatización de Reportes Académicos  
**Autora:** Lic. Ligia Elena Herrera Frías  
**Institución:** Ministerio de Educación de la República Dominicana (MINERD)  
**Entorno de Desarrollo:** Google Antigravity (AGY) IDE  

---

### Objetivo del Cuaderno:
1. Demostrar la interacción con la base de datos relacional del centro escolar (`edureport.db`).
2. Analizar la distribución de los **116 estudiantes oficiales** por ciclos, grados y secciones.
3. Verificar de manera reproducible la **Regla de las 3 Ausencias** (alerta temprana de ausentismo).
4. Evaluar el modelo de calificaciones por períodos (P1 a P4) y recuperación pedagógica (RP).
5. Evidenciar la consistencia de los datos para la generación automatizada de reportes oficiales en PDF.

In [1]:
import sqlite3
import os

# Localizar la base de datos de EduReport
db_path = os.path.join('database', 'edureport.db')
if not os.path.exists(db_path):
    db_path = 'edureport.db'

conn = sqlite3.connect(db_path)
cursor = conn.cursor()
print(f"Conectado exitosamente a la base de datos: {db_path}")

## 1. Exploración de Tablas del Sistema
EduReport cuenta con un modelo relacional de 13 entidades normalizadas.

In [2]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
tablas = cursor.fetchall()
print(f"Total de tablas en el esquema: {len(tablas)}")
for i, (nombre,) in enumerate(tablas, 1):
    cursor.execute(f"SELECT COUNT(*) FROM {nombre};")
    total = cursor.fetchone()[0]
    print(f"  {i:2d}. {nombre:<25} : {total:>5} registros")

## 2. Distribución de Estudiantes por Ciclo y Grado
Análisis de los estudiantes matriculados en el centro escolar (Primer Ciclo: 1.º a 3.º; Segundo Ciclo: 4.º a 6.º de Secundaria).

In [3]:
query = """
SELECT c.nombre AS ciclo, g.nombre AS grado, a.seccion, COUNT(e.id) AS total_estudiantes
FROM estudiantes e
JOIN aulas a ON e.aula_id = a.id
JOIN grade_levels g ON a.grade_level_id = g.id
JOIN cycles c ON g.cycle_id = c.id
GROUP BY c.id, g.id, a.id
ORDER BY c.id, g.orden;
"""

cursor.execute(query)
filas = cursor.fetchall()

print(f"{'CICLO':<18} | {'GRADO':<22} | {'SECCIÓN':<8} | {'ESTUDIANTES'}")
print("-" * 65)
total_general = 0
for ciclo, grado, seccion, total in filas:
    print(f"{ciclo:<18} | {grado:<22} | {seccion:<8} | {total:>10}")
    total_general += total
print("-" * 65)
print(f"{'TOTAL GENERAL':<44} | {total_general:>10}")

## 3. Demostración de la Regla de Negocio: Alerta de 3 Ausencias
El sistema activa automáticamente una alerta de ausentismo crítico cuando un estudiante acumula **3 o más ausencias** no justificadas.

In [4]:
# Consultar el total de ausencias por estudiante
query_ausencias = """
SELECT e.nombres, e.apellidos, e.numero_identificacion, COUNT(a.id) AS total_ausencias
FROM estudiantes e
JOIN asistencias a ON e.id = a.estudiante_id
WHERE a.estado = 'A'
GROUP BY e.id
HAVING COUNT(a.id) >= 3
ORDER BY total_ausencias DESC;
"""

cursor.execute(query_ausencias)
alertas = cursor.fetchall()

print(f"🚨 Estudiantes en ALERTA CRÍTICA (>= 3 Ausencias): {len(alertas)}")
print("-" * 70)
for nombre, apellido, rne, faltas in alertas:
    print(f"  ⚠️  {nombre} {apellido:<20} | RNE: {rne} | Faltas: {faltas} -> [Acción: Informe Generado]")

## 4. Análisis de Rendimiento Académico y Recuperación Pedagógica (RP)
Criterio MINERD: Aprobado >= 70 puntos. Si la nota inicial es menor a 70, aplica Recuperación Pedagógica.

In [5]:
query_eval = """
SELECT 
    COUNT(*) as total_evaluaciones,
    AVG(calificacion) as promedio_general,
    SUM(CASE WHEN calificacion >= 70 THEN 1 ELSE 0 END) as aprobados,
    SUM(CASE WHEN calificacion < 70 THEN 1 ELSE 0 END) as requieren_rp,
    SUM(CASE WHEN es_recuperacion = 1 THEN 1 ELSE 0 END) as en_recuperacion
FROM evaluaciones;
"""

cursor.execute(query_eval)
total_ev, prom, aprob, req_rp, en_rp = cursor.fetchone()
prom = prom or 0.0

print(f"Resumen de Calificaciones Registradas:")
print(f"  - Total de Evaluaciones: {total_ev}")
print(f"  - Promedio General:      {prom:.2f} / 100 pts")
print(f"  - Aprobados (>= 70):     {aprob} ({((aprob/total_ev)*100 if total_ev else 0):.1f}%)")
print(f"  - Requieren RP (< 70):   {req_rp}")
print(f"  - Casos en Recuperación: {en_rp}")

## 5. Reportes Generados y Trazabilidad Institucional
Registro de los reportes emitidos listos para descarga y firma oficial.

In [6]:
cursor.execute("SELECT tipo_reporte, COUNT(*) FROM reportes GROUP BY tipo_reporte;")
reportes = cursor.fetchall()

print("Distribución de Reportes Generados por Tipo:")
for tipo, cant in reportes:
    print(f"  📄 {tipo:<30}: {cant:>3} reportes")

conn.close()
print("\nVerificación y análisis completados exitosamente en Antigravity IDE.")